# 01 — Data Understanding

Proyecto **Coffee Rewards Offers** (Maven Analytics).

Objetivo de este notebook: establecer una línea de base del estado real de los
datos crudos **antes** de limpiar. Documentamos esquema, dimensiones, tipos,
nulos y problemas de calidad por tabla. No se transforma nada aquí: las
decisiones que surgen se aplican recién en la Etapa 2 (Data Cleaning).

Stack permitido: Python + Pandas, SQL, Excel. Sin ML, sin dashboards.


## Setup

In [1]:
import pandas as pd

RAW = "data/raw"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)
print(f"pandas {pd.__version__}")

pandas 3.0.5


## Carga de las 4 tablas

In [2]:
customers = pd.read_csv(f"{RAW}/customers.csv")
offers    = pd.read_csv(f"{RAW}/offers.csv")
events    = pd.read_csv(f"{RAW}/events.csv")
dictionary = pd.read_csv(f"{RAW}/data_dictionary.csv")

print("customers:", customers.shape)
print("offers:   ", offers.shape)
print("events:   ", events.shape)
print("dictionary:", dictionary.shape)

customers: (17000, 5)
offers:    (10, 6)
events:    (306534, 4)
dictionary: (18, 3)


## Resumen de dimensiones y columnas

In [3]:
overview = {
    "customers.csv": list(customers.columns),
    "offers.csv":    list(offers.columns),
    "events.csv":    list(events.columns),
    "data_dictionary.csv": list(dictionary.columns),
}
for name, cols in overview.items():
    print(f"{name:20s} -> {cols}")

customers.csv        -> ['customer_id', 'became_member_on', 'gender', 'age', 'income']
offers.csv           -> ['offer_id', 'offer_type', 'difficulty', 'reward', 'duration', 'channels']
events.csv           -> ['customer_id', 'event', 'value', 'time']
data_dictionary.csv  -> ['Table', 'Field', 'Description']


---

## 1) `customers.csv` — demografía de miembros

Columnas: `customer_id`, `became_member_on`, `gender`, `age`, `income`.


In [4]:
customers.head(8)

,customer_id,became_member_on,gender,age,income
0,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN,118,NaN
1,0610b486422d4921ae7d2bf64640c50b,20170715,F,55,112000.0
2,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN,118,NaN
3,78afa995795e4d85b5d9ceeca43f5fef,20170509,F,75,100000.0
4,a03223e636434f42ac4c3df47e8bac43,20170804,NaN,118,NaN
5,e2127556f4f64592b11af22de27a7932,20180426,M,68,70000.0
6,8ec6ce2a7e7949b1bf142def7d0e0586,20170925,NaN,118,NaN
7,68617ca6246f4fbc85e91a2a49552598,20171002,NaN,118,NaN


In [5]:
customers.dtypes

customer_id             str
became_member_on      int64
gender                  str
age                   int64
income              float64
dtype: object

### Nulos y cardinalidad

In [6]:
print("Nulos por columna:")
print(customers.isna().sum().to_string())
print()
print("Nunique por columna:")
print(customers.nunique().to_string())

Nulos por columna:


customer_id            0
became_member_on       0
gender              2175
age                    0
income              2175

Nunique por columna:
customer_id         17000
became_member_on     1716
gender                  3
age                    85
income                 91


### Hallazgo clave: `age=118` es un sentinel de datos faltantes

`age=118` aparece **2175 veces** y coincide **exactamente** con las filas que
tienen `gender` e `income` vacíos. Es decir, hay una única cohorte de 2175
registros sin datos demográficos, enmascarada con `age=118`.


In [7]:
cohort = customers[customers["age"] == 118]
print("filas con age==118:", len(cohort))
print("  -> gender faltante:", cohort["gender"].isna().sum(), "/", len(cohort))
print("  -> income faltante:", cohort["income"].isna().sum(), "/", len(cohort))

real_age = customers.loc[customers["age"] != 118, "age"]
print("edad real (sin sentinel): min =", real_age.min(), " max =", real_age.max())

filas con age==118: 2175
  -> gender faltante: 2175 / 2175
  -> income faltante: 2175 / 2175
edad real (sin sentinel): min = 18  max = 101


In [8]:
print("gender value counts:")
print(customers["gender"].value_counts(dropna=False).to_string())
print()
print("became_member_on: min =", customers["became_member_on"].min(),
      " max =", customers["became_member_on"].max())

gender value counts:
gender
M      8484
F      6129
NaN    2175
O       212

became_member_on: min = 20130729  max = 20180726


In [9]:
customers["income"].dropna().describe().round(0)

count     14825.0
mean      65405.0
std       21598.0
min       30000.0
25%       49000.0
50%       64000.0
75%       80000.0
max      120000.0
Name: income, dtype: float64

---

## 2) `offers.csv` — catálogo de ofertas (10 ofertas)

Columnas: `offer_id`, `offer_type`, `difficulty`, `reward`, `duration`, `channels`.


In [10]:
offers

,offer_id,offer_type,difficulty,reward,duration,channels
0,ae264e3637204a6fb9bb56bc8210ddfd,bogo,10,10,7,"['email', 'mobile', 'social']"
1,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10,10,5,"['web', 'email', 'mobile', 'social']"
2,3f207df678b143eea3cee63160fa8bed,informational,0,0,4,"['web', 'email', 'mobile']"
3,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5,5,7,"['web', 'email', 'mobile']"
4,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,20,5,10,"['web', 'email']"
5,2298d6c36e964ae4a3e7e9706d1fb8c2,discount,7,3,7,"['web', 'email', 'mobile', 'social']"
6,fafdcd668e3743c1bb461111dcafc2a4,discount,10,2,10,"['web', 'email', 'mobile', 'social']"
7,5a8bc65990b245e5a138643cd4eb9837,informational,0,0,3,"['email', 'mobile', 'social']"
8,f19421c1d4aa40978ebb69ca19b0e20d,bogo,5,5,5,"['web', 'email', 'mobile', 'social']"
9,2906b810c7d4411798c6938adc9daaa5,discount,10,2,7,"['web', 'email', 'mobile']"


In [11]:
print("Nulos por columna:")
print(offers.isna().sum().to_string())
print()
print("offer_type value counts:")
print(offers["offer_type"].value_counts().to_string())

Nulos por columna:
offer_id      0
offer_type    0
difficulty    0
reward        0
duration      0
channels      0

offer_type value counts:
offer_type
bogo             4
discount         4
informational    2


### Hallazgo: `channels` guardado como literal de lista Python

`channels` es un string con formato de lista Python (`['email', 'mobile', ...]`).
Se parseará con `ast.literal_eval` en la Etapa 2.


In [12]:
print("muestra de channels:")
for v in offers["channels"].head(3):
    print("  ", v)

muestra de channels:
   ['email', 'mobile', 'social']
   ['web', 'email', 'mobile', 'social']
   ['web', 'email', 'mobile']


---

## 3) `events.csv` — actividad de clientes (306.534 eventos)

Columnas: `customer_id`, `event`, `value`, `time`.


In [13]:
events.head(8)

,customer_id,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0
5,389bc3fa690240e798340f5a15918d5c,offer received,{'offer id': 'f19421c1d4aa40978ebb69ca19b0e20d'},0
6,c4863c7985cf408faee930f111475da3,offer received,{'offer id': '2298d6c36e964ae4a3e7e9706d1fb8c2'},0
7,2eeac8d8feae4a8cad5a6af0499a211d,offer received,{'offer id': '3f207df678b143eea3cee63160fa8bed'},0


In [14]:
print("Nulos por columna:")
print(events.isna().sum().to_string())
print()
print("event value counts:")
print(events["event"].value_counts().to_string())

Nulos por columna:
customer_id    0
event          0
value          0
time           0

event value counts:
event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579


In [15]:
print("time: min =", events["time"].min(), " max =", events["time"].max(),
      " (horas;", round(events["time"].max()/24, 2), "días)")
print("customer_id unique:", events["customer_id"].nunique())

time: min = 0  max = 714  (horas; 29.75 días)
customer_id unique: 17000


### Hallazgo: `value` guardado como literal de dict Python

`value` cambia de estructura según el tipo de evento:
- `transaction` -> `{'amount': ...}`
- `offer received` / `offer viewed` -> `{'offer id': ...}`
- `offer completed` -> `{'offer_id': ..., 'reward': ...}`


In [16]:
for e in ["transaction", "offer received", "offer completed"]:
    sub = events[events["event"] == e]
    print(f"{e:18s} n={len(sub):6d}  sample:", sub["value"].head(2).to_list())

transaction        n=138953  sample: ["{'amount': 0.8300000000000001}", "{'amount': 34.56}"]
offer received     n= 76277  sample: ["{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}", "{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}"]
offer completed    n= 33579  sample: ["{'offer_id': '2906b810c7d4411798c6938adc9daaa5', 'reward': 2}", "{'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4', 'reward': 2}"]


---

## 4) `data_dictionary.csv` — diccionario de datos

Documenta tablas y campos (18 filas).


In [17]:
dictionary

,Table,Field,Description
0,offers,NaN,Details on the offers sent to customers during...
1,offers,offer_id,Unique offer ID (primary key)
2,offers,offer_type,"type of offer: bogo (buy one, get one), discou..."
3,offers,difficulty,minimum amount required to spend in order to b...
4,offers,reward,reward (in dollars) obtained by completing the...
5,offers,duration,days a customer has to complete the offer once...
6,offers,channels,list of marketing channels used to send the of...
7,customers,NaN,Demographic data for each member
8,customers,customer_id,Unique customer ID (primary key)
9,customers,became_member_on,Date when the customer created their account (...


---

## Resumen de problemas de calidad detectados

| # | Tabla | Problema | Decisión (Etapa 2) |
|---|-------|----------|--------------------|
| 1 | customers | `age=118` = sentinel de faltante (2175 filas, misma cohorte con `gender`/`income` vacíos) | reemplazar por `NaN` |
| 2 | customers | `gender` vacío en 2175 filas | dejar como `NaN` (missing) |
| 3 | customers | `income` vacío en 2175 filas | dejar como `NaN` (missing) |
| 4 | customers | `became_member_on` leído como `int` (`yyyymmdd`) | convertir a `datetime` |
| 5 | offers | `channels` como literal de lista Python | parsear con `ast.literal_eval` |
| 6 | events | `value` como literal de dict Python | parsear con `ast.literal_eval` y extraer `offer_id` / `amount` / `reward` |

## Línea de base (cifras clave)

- `customers`: 17.000 filas, 5 columnas. 2175 sin datos demográficos.
- `offers`: 10 ofertas (4 bogo, 4 discount, 2 informational).
- `events`: 306.534 eventos (transaction 138.953, offer received 76.277,
  offer viewed 57.725, offer completed 33.579). Ventana temporal ~30 días
  (0 a 714 horas).
